# PreNormLoop-0.6B 模型结构

**18 个独立 Transformer 块，循环 3 轮，共执行 54 次。** 先看整体路径，再从模型对象里读模块、线性层尺寸和跨轮数据流。

Notebook 只读取固定版本的配置与 Python 源码。模型骨架在 `meta` 设备上构建，分配的模型权重存储为 **0 字节**。文中的张量形状由配置和模块属性推导。 [模型仓库](https://huggingface.co/osunlp/PreNormLoop-0.6B) · [发布者实现](https://github.com/OSU-NLP-Group/ARLoop)

## 1. 先看整体结构

Token ID → Token Embedding → **同一个 18 层共享栈 ×3** → 最后一次 RMSNorm → Logits

每个块有两次残差更新，分别由 RMSNorm → Attention 和 RMSNorm → MLP 产生。目标是看清一轮结束后，隐藏状态怎样流入下一轮。 [发布配置](https://huggingface.co/osunlp/PreNormLoop-0.6B/blob/e4ad59d0d03ab78f83d9f52b0c7d7267f372a151/config.json)

In [1]:
import hashlib
import json
from urllib.request import Request, urlopen

from IPython.display import Markdown, display

revision = "e4ad59d0d03ab78f83d9f52b0c7d7267f372a151"
repo = "https://huggingface.co/osunlp/PreNormLoop-0.6B"
files = {
    "config.json": "1e0b2e41b5ca188cdbddc8c8313d7f2b84c49dbf9772c1ca6626be4efedc78ea",
    "configuration_baseline.py": "f1cac2589a855c9e99015957a766f75fa854b7f28928c1dcfe0aab5cc126a092",
    "modeling_baseline.py": "ed5139ce9497369f5263ceb58ca913042c8bde9fd1193ff71135f1116359c969",
}


def read_public_text(filename):
    expected_sha256 = files[filename]
    request = Request(
        f"{repo}/raw/{revision}/{filename}",
        headers={"User-Agent": "Jupyter architecture inspection/1.0"},
    )
    with urlopen(request, timeout=25) as response:
        data = response.read(64 * 1024 + 1)
    if len(data) > 64 * 1024 or hashlib.sha256(data).hexdigest() != expected_sha256:
        raise ValueError(f"文件大小或 SHA-256 校验失败：{filename}")
    print(f"{filename:22} {len(data):,} bytes")
    return data.decode("utf-8")


config_text = read_public_text("config.json")
cfg = json.loads(config_text)
layer_count = cfg["physical_layers"]
loop_count = cfg["loop_count"]
effective_depth = layer_count * loop_count
assert effective_depth == cfg["num_hidden_layers"]

rows = [
    ("独立 Transformer 层", "physical_layers"),
    ("循环次数", "loop_count"),
    ("有效层计算次数", "num_hidden_layers"),
    ("Hidden Size", "hidden_size"),
    ("Attention Query Heads", "num_attention_heads"),
    ("Attention KV Heads", "num_key_value_heads"),
    ("每个 Attention Head 的维度", "head_dim"),
    ("MLP 中间维度", "intermediate_size"),
]
table = ["| 读什么 | 配置字段 | 值 |", "|---|---|---:|"]
table.extend(f"| {label} | `{key}` | {cfg[key]:,} |" for label, key in rows)
display(Markdown("\n".join(table)))

config.json            861 bytes


| 读什么 | 配置字段 | 值 |
|---|---|---:|
| 独立 Transformer 层 | `physical_layers` | 18 |
| 循环次数 | `loop_count` | 3 |
| 有效层计算次数 | `num_hidden_layers` | 54 |
| Hidden Size | `hidden_size` | 1,536 |
| Attention Query Heads | `num_attention_heads` | 16 |
| Attention KV Heads | `num_key_value_heads` | 4 |
| 每个 Attention Head 的维度 | `head_dim` | 96 |
| MLP 中间维度 | `intermediate_size` | 4,608 |

这里 `num_hidden_layers` 表示展开后的执行深度。接下来回到构造代码，看看真正创建了多少个块。

## 2. 查看真实模块树

用发布者定义的 `ARLoopBaselineForCausalLM` 读取实际模块。`model.transformer` 是 `ModuleDict`，包含 Token Embedding `wte`、18 层共享栈 `h` 和最终的 `ln_f`。`shared_value_embedding` 挂在模型根节点，供有 Value 门控的层使用。

| 路径 | 在模型里的职责 |
|---|---|
| `model.transformer.wte` | Token ID → 1,536 维向量，输出投影也读取这张权重表 |
| `model.transformer.h` | 18 个 `PreNormBlock`，每轮按同一顺序调用 |
| `model.transformer.ln_f` | 所有循环结束后，对最后的隐藏状态做 RMSNorm |
| `model.shared_value_embedding` | Token ID → 384 维的 Value 向量，供部分 Attention 层使用 |

构建用 `AutoConfig` 和 `AutoModelForCausalLM.from_config`。`trust_remote_code=True` 载入的是固定版本、经哈希校验的本地 Python 文件，`no_init_weights()` 跳过权重初始化。下面先核对 18 层，再展开完整模块树。 [构造源码](https://huggingface.co/osunlp/PreNormLoop-0.6B/blob/e4ad59d0d03ab78f83d9f52b0c7d7267f372a151/modeling_baseline.py#L103)

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
import transformers
from transformers import AutoConfig, AutoModelForCausalLM
from transformers.initialization import no_init_weights

source_files = {
    "config.json": config_text,
    "configuration_baseline.py": read_public_text("configuration_baseline.py"),
    "modeling_baseline.py": read_public_text("modeling_baseline.py"),
}

with TemporaryDirectory(prefix="prenormloop_code_") as directory:
    source_dir = Path(directory)
    for name, content in source_files.items():
        (source_dir / name).write_text(content, encoding="utf-8")

    model_config = AutoConfig.from_pretrained(
        source_dir, local_files_only=True, trust_remote_code=True
    )
    assert Path(model_config.name_or_path) == source_dir
    rng_before = torch.get_rng_state()
    with torch.device("meta"), no_init_weights():
        model = AutoModelForCausalLM.from_config(model_config, trust_remote_code=True)
    assert torch.equal(rng_before, torch.get_rng_state())

parameters = list(model.parameters())
assert parameters and all(
    parameter.is_meta and parameter.untyped_storage().data_ptr() == 0
    for parameter in parameters
)
assert all(buffer.is_meta for buffer in model.buffers())
assert type(model).__name__ == cfg["architectures"][0]

print(f"PyTorch {torch.__version__} · Transformers {transformers.__version__}")
print(f"模型类型：{type(model).__name__}")
print(f"逻辑参数量：{sum(parameter.numel() for parameter in parameters):,}")
print(f"{len(parameters)} 个参数张量均在 meta，实际参数存储为 0 字节")
print("CPU 随机数状态未变")

configuration_baseline.py 1,909 bytes


modeling_baseline.py   10,004 bytes


PyTorch 2.15.0 · Transformers 5.22.0
模型类型：ARLoopBaselineForCausalLM
逻辑参数量：582,804,096
156 个参数张量均在 meta，实际参数存储为 0 字节
CPU 随机数状态未变


In [3]:
shared_layers = model.transformer.h
assert isinstance(shared_layers, torch.nn.ModuleList)
assert len(shared_layers) == layer_count == 18
assert len({id(layer) for layer in shared_layers}) == 18

print(f"model.transformer.h：{type(shared_layers).__name__}，{len(shared_layers)} 层")
for index, layer in enumerate(shared_layers):
    print(f"  h[{index:2d}]  {type(layer).__name__}")

model.transformer.h：ModuleList，18 层
  h[ 0]  PreNormBlock
  h[ 1]  PreNormBlock
  h[ 2]  PreNormBlock
  h[ 3]  PreNormBlock
  h[ 4]  PreNormBlock
  h[ 5]  PreNormBlock
  h[ 6]  PreNormBlock
  h[ 7]  PreNormBlock
  h[ 8]  PreNormBlock
  h[ 9]  PreNormBlock
  h[10]  PreNormBlock
  h[11]  PreNormBlock
  h[12]  PreNormBlock
  h[13]  PreNormBlock
  h[14]  PreNormBlock
  h[15]  PreNormBlock
  h[16]  PreNormBlock
  h[17]  PreNormBlock


In [4]:
print(model)

ARLoopBaselineForCausalLM(
  (transformer): ModuleDict(
    (wte): Embedding(49152, 1536)
    (h): ModuleList(
      (0): PreNormBlock(
        (norm_1): RMSNorm()
        (attn): Attention(
          (c_q): Linear(in_features=1536, out_features=1536, bias=False)
          (c_k): Linear(in_features=1536, out_features=384, bias=False)
          (c_v): Linear(in_features=1536, out_features=384, bias=False)
          (c_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (norm_2): RMSNorm()
        (mlp): MLP(
          (fc): Linear(in_features=1536, out_features=9216, bias=False)
          (proj): Linear(in_features=4608, out_features=1536, bias=False)
        )
      )
      (1): PreNormBlock(
        (norm_1): RMSNorm()
        (attn): Attention(
          (c_q): Linear(in_features=1536, out_features=1536, bias=False)
          (c_k): Linear(in_features=1536, out_features=384, bias=False)
          (c_v): Linear(in_features=1536, out_features=384, bias=Fals

## 3. 拆开一个 PreNormBlock

以 `model.transformer.h[0]` 为例。输入和输出都保持 `[B, T, 1536]`，其中 `B` 是批大小，`T` 是序列长度。两次残差更新的顺序很重要，MLP 读的是 Attention 已更新过的状态。

~~~text
u = h + Attention(RMSNorm₁(h))
h_next = u + MLP(RMSNorm₂(u))
~~~

- **Attention** 使用 GQA。Q 有 16 个头，K 和 V 各有 4 个头，每头 96 维。Q 投影到 1,536 维，K/V 各投影到 384 维。每组 4 个 Q 头共享一个 KV 头。
- **MLP** 把 1,536 维投影到 9,216 维，拆成两个 4,608 维分支。门控分支经 SiLU 后与内容分支逐元素相乘，最后投影回 1,536 维。
- **Value 门控** 出现在 `h[1]`、`h[3]` 等层。它根据当前状态对共享 Value 向量做缩放，再把结果加到 V 上。

下面的尺寸直接读取真实模块的 `in_features` 和 `out_features`。 [Attention 与 MLP 源码](https://huggingface.co/osunlp/PreNormLoop-0.6B/blob/e4ad59d0d03ab78f83d9f52b0c7d7267f372a151/modeling_baseline.py#L27)

In [5]:
block = shared_layers[0]
linears = [
    ("Attention · Q", block.attn.c_q),
    ("Attention · K", block.attn.c_k),
    ("Attention · V", block.attn.c_v),
    ("Attention · 输出投影", block.attn.c_proj),
    ("MLP · 门控和内容分支", block.mlp.fc),
    ("MLP · 输出投影", block.mlp.proj),
]
table = ["| 投影 | 输入维度 | 输出维度 |", "|---|---:|---:|"]
for label, layer in linears:
    table.append(f"| {label} | {layer.in_features:,} | {layer.out_features:,} |")
display(Markdown("\n".join(table)))

gated = [i for i, layer in enumerate(shared_layers) if layer.attn.ve_gate is not None]
print("带 Value 门控的层索引", gated)
print(f"Q / KV：{model.config.num_attention_heads} / "
      f"{model.config.num_key_value_heads} 个头，单头 {model.config.head_dim} 维")

| 投影 | 输入维度 | 输出维度 |
|---|---:|---:|
| Attention · Q | 1,536 | 1,536 |
| Attention · K | 1,536 | 384 |
| Attention · V | 1,536 | 384 |
| Attention · 输出投影 | 1,536 | 1,536 |
| MLP · 门控和内容分支 | 1,536 | 9,216 |
| MLP · 输出投影 | 4,608 | 1,536 |

带 Value 门控的层索引 [1, 3, 5, 7, 9, 11, 13, 15, 17]
Q / KV：16 / 4 个头，单头 96 维


## 4. 三轮怎么接起来

下面是 `forward` 第 163–172 行的原始源码，移除了片段外层的统一缩进。局部变量 `h` 保存隐藏状态，`self.transformer.h` 保存 18 个可重复调用的模块。 [查看发布者源码](https://huggingface.co/osunlp/PreNormLoop-0.6B/blob/e4ad59d0d03ab78f83d9f52b0c7d7267f372a151/modeling_baseline.py#L163-L172)

~~~python
h = self.transformer.wte(input_ids) * self.config.embedding_scale
ve = self.shared_value_embedding(input_ids)
states = [h] if output_hidden_states else None
for loop in range(self.config.loop_count):
    for i, block in enumerate(self.transformer.h):
        effective = loop * self.config.physical_layers + i
        h = h + block.attn(block.norm_1(h), position_ids, mask, ve, cache, effective)
        h = h + block.mlp(block.norm_2(h))
        if states is not None:
            states.append(h)
~~~

### 逐行读循环

| 源码行 | 这行做什么 |
|---:|---|
| 163 | 输入 Token ID 经过 Embedding 和缩放，得到初始隐藏状态 `h`。这行在两个循环之前，只执行一次。 |
| 164 | 为输入 Token 计算共享的 Value Embedding `ve`，后续传给各层 Attention。 |
| 165 | 如果调用者要求返回中间状态，就用初始 `h` 建一个列表。 |
| 166 | 外层按配置运行 3 轮，`loop` 依次为 0、1、2。 |
| 167 | 每轮都遍历同一个 `ModuleList`，`i` 从 0 到 17。`block` 是当前层模块。 |
| 168 | 计算从 0 开始的执行编号 `effective`。启用 KV Cache 时，Attention 用它标识本次层执行的缓存。 |
| 169 | 对当前 `h` 做 `norm_1`，计算 Attention，再把结果加回 `h` 并写回。 |
| 170 | 对 Attention 更新后的 `h` 做 `norm_2`，计算 MLP，再把结果加回并写回。这次写回得到当前层的最终输出。 |
| 171 | 检查是否需要收集中间状态。 |
| 172 | 需要时就把本层最终输出追加到列表，下一层仍直接接收当前 `h`。 |

### 第一轮最后一层怎样接到第二轮第一层

| 交接位置 | 当前模块与编号 | `h` 在这里是什么 |
|---|---|---|
| **第一轮最后一层跑完** | `loop = 0`、`i = 17`、`effective = 17`，模块是 `self.transformer.h[17]` | 第 170 行刚把第一轮第 18 层的最终输出写回 `h`。 |
| **第二轮第一层开始** | `loop = 1`、`i = 0`、`effective = 18`，模块回到 `self.transformer.h[0]` | 第 169 行右侧的 `block.norm_1(h)` 读到的就是上一行表格里的同一个隐藏状态。 |

**第一轮第 18 层在第 170 行写回 `h` → 外层进入第二轮 → 第二轮第 1 层在第 169 行读取这个 `h`。** 换轮时会重新取层模块，`h` 继续沿用。循环之间也没有再次执行第 163 行的 Embedding 初始化。

## 5. 输出头和 KV Cache 的编号

第 3 轮结束后，`ln_f` 对最终的隐藏状态做 RMSNorm。随后 `F.linear(h, self.transformer.wte.weight)` 用输入 Embedding 的权重算出每个位置对 49,152 个词表项的 Logits。这里可以用模块树对照 `wte` 的形状 `[49152, 1536]`。

源码还有一个 `effective = loop * 18 + i`。开启 KV Cache 时，Attention 用它索引本次执行的缓存。第一层模块 `h[0]` 跑三次，用到的缓存索引分别是 `0、18、36`。模块树仍然只有 18 个块。 [输出实现](https://huggingface.co/osunlp/PreNormLoop-0.6B/blob/e4ad59d0d03ab78f83d9f52b0c7d7267f372a151/modeling_baseline.py#L173) · [KV Cache 实现](https://huggingface.co/osunlp/PreNormLoop-0.6B/blob/e4ad59d0d03ab78f83d9f52b0c7d7267f372a151/modeling_baseline.py#L64)